## Скрипт выгрузки данных Росстата по промпроизводству в обработке из отдельных помесячных файлов

In [1]:
# Выгрузим библиотеки
import pandas as pd
import requests
import os
from io import BytesIO
from datetime import datetime
from dateutil.relativedelta import relativedelta
from dotenv import load_dotenv
load_dotenv('data.env')

True

In [2]:
# Загрузим наш шаблон
df_monitor = pd.read_excel(os.getenv('MASK_FILE'))
df_monitor = df_monitor.iloc[:, 0:3]
df_monitor = df_monitor.dropna(how='all')
df_monitor.columns = df_monitor.iloc[0]
df_monitor = df_monitor.iloc[1:].reset_index(drop=True)
df_monitor = df_monitor.dropna(subset=['Код ОКПД 2'])
df_monitor['Код ОКПД 2'] = df_monitor['Код ОКПД 2'].astype(str)
df_monitor['v_okved'] = df_monitor['Код ОКПД 2'].str[0:2]
display(df_monitor.head(5))

1,Код ОКПД 2,Товар,ед.изм.,v_okved
0,26.11.3,Схемы интегральные электронные,Тысяча штук,26
1,26.2,Компьютеры и периферийное оборудование,Тысяча рублей,26
2,28.30.2,Тракторы для сельского хозяйства прочие,Штука,28
3,28.30.59.111,Комбайны зерноуборочные,Штука,28
4,28.41.11.001.АГ,Станки металлорежущие,Штука,28


In [3]:
# Функция выгрузки значения по заданному коду в заданном датафрейме
import re
pattern = re.compile(r'^\d{2}\.\d{2}\.\d{2}')
def match_mask(value):
    if not isinstance(value, str):
        return False
    else:
        return bool(pattern.match(value))

def value_okpd(okpd, mes, df):
    temp_value=None
    try:
        temp_index = df.index[df['Unnamed: 1']==str(okpd)].tolist()
        
        # Формируем таблицу для этого ОКПД
        if temp_index:      
            df_start = temp_index[0]
            df_end = df_start+1
            counter = 0
            while counter<=20:
                if df_end >= len(df):
                    df_end = len(df)
                    break
                if match_mask(df.iloc[df_end]['Unnamed: 1']):
                    df_end = df_end - 1
                    break
                counter += 1
                df_end += 1
            target_df = df.iloc[df_start:df_end].copy()
            # print(f"start/end: {df_start}, {df_end}")
            # Ищем mes
            mes_index = target_df.index[target_df.iloc[:,0] == str(mes)].to_list()
            # print(f"mes: {mes_index}")
            if mes_index:  # Проверяем, что список не пустой
                temp_value = target_df.loc[mes_index[0] + 1]['Unnamed: 2']
                return temp_value
        else:
            print(f"Код {okpd} не найден в файле")
            if len(okpd)==7:
                print('Проверяем на 7 знаков')
                okpd = okpd + '0'
                temp_index = df.index[df['Unnamed: 1']==str(okpd)].tolist()
                if temp_index: 
                    df_start = temp_index[0]
                    df_end = df_start+1
                    counter = 0
                    while counter<=20:
                        if df_end >= len(df):
                            df_end = len(df)
                            break
                        if match_mask(df.iloc[df_end]['Unnamed: 1']):
                            df_end = df_end - 1
                            break
                        counter += 1
                        df_end += 1
                target_df = df.iloc[df_start:df_end].copy()
                mes_index = target_df.index[target_df.iloc[:,0] == str(mes)].to_list()
                # print(f"mes: {mes_index}")
                if mes_index:
                    temp_value = target_df.loc[mes_index[0] + 1]['Unnamed: 2']
                    return temp_value
                else:
                    print(f"Код {okpd} (с дополненным 0) не найден в файле")
    except IndexError:
        print('Тест ошибки с индексом')
    except Exception as e:
        print(f"Ошибка при обработке данных: {type(e).__name__}: {e}")
    return temp_value

In [4]:
# Функция выгрузки данных в соответствии с заданной структурой (месяцы-классы)
def fetch_data(months_dict):
    df_dict = {}
    for month in months_dict:
        try:
            print(f"Загрузка данных за период {month}")
            file_name = f"http://ssl.rosstat.gov.ru/storage/mediabank/Prom_{month}.xlsx"
            file_handshake = requests.head(file_name, verify=False)
            if file_handshake.status_code == 200:
                file_response = requests.get(file_name, verify=False, timeout=30)
                file_excel = BytesIO(file_response.content)
            else:
                print(f"Ошибка получения данных. Статус: {file_handshake.status_code}. Данные могут отсутствовать.")
                global month_not_exist
                month_not_exist=month
                print(f"Последний месяц{month}")
                break
            for code in months_dict.get(month):
                print(f"Загрузка данных по классу: {code}")
                sheet_combo = f"Н-П_{code}"        
                df_dict[f"{month}_{code}"] = pd.read_excel(file_excel, sheet_name=sheet_combo)
    
        except requests.exceptions.RequestException as e:
            print(f"Произошла ошибка при выполнении запроса: {e}")
        except requests.exceptions.Timeout:
            print(f"✗ Таймаут при проверке {date_str}")
        except requests.exceptions.ConnectionError:
            print(f"✗ Ошибка подключения")
        except Exception as e:
            print(f"✗ Ошибка: {type(e).__name__}: {e}")
    return df_dict


In [5]:
# Функция создания структуры месяц-класс в соответствии с шаблоном и периодом загрузки
def get_month_dict(periods, mask_df):
    current_date = datetime.now()
    some_time_ago = current_date - relativedelta(months=periods)
    start_date = some_time_ago.replace(day=1)  # Первое число месяца
    
    dict_data={}
    temp_value=None
    
    # Генерируем даты по месяцам до текущего месяца
    while start_date <= current_date.replace(day=1):  # Только до текущего месяца
        month_str = start_date.strftime("%m_%Y")
        dict_data[month_str] = pd.Series(mask_df['v_okved'].dropna().unique()).sort_values().tolist()
        # Переходим к следующему месяцу
        start_date = start_date + relativedelta(months=1)
    return dict_data

In [6]:
# Создаем структуру дата-класс и загружаем данные
find_periods = int(os.getenv('PERIODS'))
months_practice=get_month_dict(find_periods, df_monitor)
global_df_dict = fetch_data(months_practice)
df_temp=pd.DataFrame(None)

Загрузка данных за период 02_2026
Загрузка данных по классу: 10
Загрузка данных по классу: 16
Загрузка данных по классу: 17
Загрузка данных по классу: 18
Загрузка данных по классу: 20
Загрузка данных по классу: 22
Загрузка данных по классу: 24
Загрузка данных по классу: 26
Загрузка данных по классу: 28
Загрузка данных по классу: 29
Загрузка данных по классу: 30
Загрузка данных за период 03_2026
Загрузка данных по классу: 10
Загрузка данных по классу: 16
Загрузка данных по классу: 17
Загрузка данных по классу: 18
Загрузка данных по классу: 20
Загрузка данных по классу: 22
Загрузка данных по классу: 24
Загрузка данных по классу: 26
Загрузка данных по классу: 28
Загрузка данных по классу: 29
Загрузка данных по классу: 30
Загрузка данных за период 04_2026
Загрузка данных по классу: 10
Загрузка данных по классу: 16
Загрузка данных по классу: 17
Загрузка данных по классу: 18
Загрузка данных по классу: 20
Загрузка данных по классу: 22
Загрузка данных по классу: 24
Загрузка данных по классу: 2

In [8]:
# Наполняем целевой датафрейм
list_temp=[]
target_df = df_monitor.copy()
for key in months_practice: 
    if key == month_not_exist:
        break
    for index, row in df_monitor.iterrows():
        code = row['Код ОКПД 2']
        ed  = row['ед.изм.']
        list_temp.append(value_okpd(code, ed, global_df_dict[f"{key}_{str(code)[0:2]}"]))
    target_df[key] = list_temp
    list_temp=[]

# Грузим датафрейм в нужный файл
target_df.to_excel(os.getenv('OUTPUT_FILE'), sheet_name='Данные', index=False)
display(target_df.head())

Код 26.11.3 не найден в файле
Проверяем на 7 знаков
Код 30.30.32.110 не найден в файле
Код 30.30.33.110 не найден в файле
Код 30.30.34.110 не найден в файле
Код 16.10.1 не найден в файле
Проверяем на 7 знаков
Код 18 не найден в файле
Код 30.20.20.140 + 30.20.32.130 не найден в файле
Код 30.20.32.130 не найден в файле
Код 2021-11-28 00:00:00 не найден в файле
Код 2022-11-28 00:00:00 не найден в файле
Код 2023-11-28 00:00:00 не найден в файле
Код 26.11.3 не найден в файле
Проверяем на 7 знаков
Код 30.30.34.110 не найден в файле
Код 16.10.1 не найден в файле
Проверяем на 7 знаков
Код 18 не найден в файле
Код 30.20.20.140 + 30.20.32.130 не найден в файле
Код 2021-11-28 00:00:00 не найден в файле
Код 2022-11-28 00:00:00 не найден в файле
Код 2023-11-28 00:00:00 не найден в файле
Код 26.11.3 не найден в файле
Проверяем на 7 знаков
Код 30.30.34.110 не найден в файле
Код 16.10.1 не найден в файле
Проверяем на 7 знаков
Код 18 не найден в файле
Код 30.20.20.140 + 30.20.32.130 не найден в файле
К

1,Код ОКПД 2,Товар,ед.изм.,v_okved,02_2026,03_2026,04_2026,05_2026
0,26.11.3,Схемы интегральные электронные,Тысяча штук,26,3.946435e+04,6.072680e+04,6.681332e+04,6.994119e+04
1,26.2,Компьютеры и периферийное оборудование,Тысяча рублей,26,2.091888e+07,2.552639e+07,2.808451e+07,2.355155e+07
2,28.30.2,Тракторы для сельского хозяйства прочие,Штука,28,3.680000e+02,4.280000e+02,5.760000e+02,3.670000e+02
3,28.30.59.111,Комбайны зерноуборочные,Штука,28,1.850000e+02,2.120000e+02,2.450000e+02,3.780000e+02
4,28.41.11.001.АГ,Станки металлорежущие,Штука,28,8.980000e+02,9.420000e+02,1.187000e+03,9.020000e+02
